# Lab 2: Train, compare, register, and use a milestone-delay model

Train six classifiers in two rounds within one AIDP experiment, compare their validation results, register the winner in the Master Catalog, reload its explicit version and score Gold milestones. Task 11 stages the Austin PDFs in a volume; knowledge-base investigation, quality/freshness review and readiness assessment are not part of this lab.

Allow 60–90 minutes plus compute startup. Experiments and Models are AIDP Preview features.

### What you will learn

Create an experiment before training; log parameters, metrics and model artifacts; compare two model families; select a run using validation data; register that run's model in a catalog schema; reload a specific version and score a Gold batch. The previously optional reload-and-score exercise is included in this lab.

| Stage | AIDP object or output |
| --- | --- |
| Both training rounds | Experiment `seer_milestone_delay` |
| Round one | Three Gradient Boosting runs: `gb_r1_1` to `gb_r1_3` |
| Round two | Three Decision Tree runs: `dt_r2_1` to `dt_r2_3` |
| Selected model | `seer_livelabs_20260922.seer_gold.milestone_delay_classifier` |
| Batch predictions | `seer_livelabs_20260922.seer_gold.milestone_delay_predictions` |
| PDF staging | Three Austin PDFs in the Gold output volume |

### How to run this lab

Use workspace **seer_livelabs_20260922**. Run one Python cell at a time, from top to bottom, and wait for completion before continuing. Task numbers are Markdown headings, not Python execution-counter numbers. There are 11 Python cells across 11 tasks. The lab ends after Task 11.

For the first walkthrough, do not choose **Run all**: stop after Task 6 to inspect the first round before training the second round. There is no automatic pause in the code. A workflow or Run all executes all 11 Python cells continuously.

## Prerequisites

1. Finish Lab 1A, Lab 1B and Lab 1C in order.
2. Verify Gold contains `project_context` and `milestone_features`.
3. Open **Shared → seer-aidp-livelabs → notebooks → Lab2_Predict_Investigate_and_Operationalize.ipynb**.
4. Attach **seer_livelabs_spark**. Use AMD/Intel compute, not ARM.
5. Verify MLflow, scikit-learn, pandas and NumPy import successfully in Task 1. Use the platform-provided MLflow integration; do not install a replacement MLflow package or point tracking to a local folder. Round one uses scikit-learn Gradient Boosting, so no XGBoost installation is required.
6. Your identity needs access to workspace experiments and permission to register model versions in `seer_livelabs_20260922.seer_gold`.

Only three Gold milestones are available and no labeled historical source exists. The training history is clearly marked **synthetic**. Scores and test metrics are educational, not validated forecasts.

The model is **trained on generated history**, then **scores the Gold feature rows created by Lab 1C**. It is not trained on the three current Gold rows or the PDFs. All model code is in the notebook; no workspace `lib` imports are needed. This revision still reads the existing `config/workshop.json`, PDF manifest—keep those supplied support files in place.

## Rerunning Lab 2

You may repeat this lab sequentially without deleting previous experiments, runs, model versions or results. A complete run from Task 1 reuses `seer_milestone_delay`, generates a new `SESSION_ID`, adds six training runs, registers another model version and overwrites `milestone_delay_predictions`. Earlier runs do not participate in the new session-filtered selection. Lab 1 tables are read, not reset.

- Run Task 1 once at the beginning, then both training rounds once each. Do not rerun Task 1 between rounds.
- Repeating Task 5 or Task 7 after runs have been created can leave extra completed candidates and fail Task 8's exact-six-run check. Fix the cause and start a fresh walkthrough from Task 1 after a partial training failure; no deletion is required.
- For scoring only, rerun Task 10 in the same active notebook session with its setup and selected-model variables still available. It loads the selected registered version and replaces predictions without retraining. It is not a standalone fresh-session cell.
- Task 9 creates another version each time. Check Models → Versions before retrying after an error; registration may already have succeeded.
- Task 11 reuses identical PDFs and copies only missing files; no knowledge base or assistant is required.
- Run one writer at a time. A full **four-task workflow** also resets Lab 1 tables; it is not equivalent to running Lab 2 alone.
- Cleanup is optional storage/history housekeeping. Retained runs and models accumulate artifacts; keep versions/source runs needed for traceability. Do not delete shared catalogs, source volumes or unrelated resources as a reset shortcut.

See the **Reruns and recovery** section of the repository's root `README.md` for restart points and failure recovery for both labs.


## Task 1: Create the experiment

1. Run the first code cell.
2. It calls `mlflow.set_experiment("seer_milestone_delay")`, creating the experiment if absent and reusing it otherwise.
3. Note the printed experiment ID and comparison-session ID.
4. Open **Experiments** in the workspace navigation.
5. Open **seer_milestone_delay**. It may have no runs until Task 5.
6. Return to the notebook. Do not change the session ID between rounds.

The cell also defines the registered model name, the nine features and a 0.5 evaluation threshold. Existing runs are retained.

**Checkpoint:** the output shows an experiment ID, a comparison-session ID and library versions. The experiment ID need not be 1 in your environment. Run this setup only once per walkthrough: rerunning it changes `SESSION_ID`, so previous candidates will not appear in the new session-filtered comparison.


In [ ]:
import os, json, hashlib, shutil, uuid
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

LAB_HOME = Path(os.environ.get("SEER_LAB_HOME", "/Workspace/Shared/seer-aidp-livelabs"))

utilities = globals().get("oidlUtils")
config_path = str(LAB_HOME / "config/workshop.json")
RUN_ID = os.environ.get("SEER_RUN_ID", "interactive")
if utilities is not None:
    config_path = utilities.parameters.getParameter("config_path", config_path)
    RUN_ID = utilities.parameters.getParameter("run_id", RUN_ID)
config = json.loads(Path(config_path).read_text())
BRONZE = f"{config['catalog']}.{config['schemas']['bronze']}"
SILVER = f"{config['catalog']}.{config['schemas']['silver']}"
GOLD = f"{config['catalog']}.{config['schemas']['gold']}"
keys = ["project_id", "asset_id", "milestone_id"]

FEATURES = ["days_to_planned_date", "open_po_fraction", "inspection_fail_flag",
            "inspection_info_flag", "supplier_review_flag", "unresolved_supplier_flag",
            "milestone_not_started_flag", "milestone_at_risk_flag", "log_committed_cost"]

# Validate the nine model inputs before training or scoring.
def feature_matrix(frame):
    values = frame[FEATURES].to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError("Missing/nonfinite feature: resolve evidence or explicitly redesign missing-value handling")
    if not frame.days_to_planned_date.between(1, 30).all() or not frame.open_po_fraction.between(0, 1).all():
        raise ValueError("Feature outside workshop prediction horizon/range")
    for name in FEATURES[2:8]:
        if not frame[name].isin([0, 1]).all():
            raise ValueError(name)
    return values


import mlflow
import mlflow.sklearn
import mlflow.pyfunc
import sklearn
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, f1_score, brier_score_loss
from mlflow.models import infer_signature

EXPERIMENT_NAME = "seer_milestone_delay"
MODEL_NAME = f"{GOLD}.milestone_delay_classifier"
experiment = mlflow.set_experiment(EXPERIMENT_NAME)
client = mlflow.MlflowClient()
# Explicit logging makes the six runs and their artifacts easy to follow.
mlflow.autolog(disable=True)
SESSION_ID = uuid.uuid4().hex
threshold = float(config["classification_threshold"])
print("Experiment:", EXPERIMENT_NAME, "| ID:", experiment.experiment_id)
print("Comparison session:", SESSION_ID)
print("MLflow:", mlflow.__version__, "| scikit-learn:", sklearn.__version__)


## Task 2: Prepare labeled teaching history

1. Run the synthetic-history cell.
2. Inspect sample milestone IDs, dates and labels.
3. Explain that `late_flag = 1` means actual completion occurred after the planned date.
4. Notice that actual completion, label availability, identifiers and labels are excluded from model inputs.
5. The seed and generator make the 1,800-row fixture reproducible. PDFs are not used here.

**Checkpoint:** five sample rows appear with `SYN-` milestone IDs and `SYNTHETIC_TRAINING_ONLY`. These are examples for training, not new Gold tables.


In [ ]:
# Convert log-odds to probabilities safely.
def sigmoid(values):
    # Clipping protects floating-point arithmetic, not a probability calibration.
    return 1. / (1. + np.exp(-np.clip(values, -35, 35)))

# Generate labeled teaching examples; this is not real project history.
def synthetic_history(seed=234582, count=1800):
    """Generate reproducible stochastic teaching data, never represented as real.

    Probabilistic outcomes depend on observed factors plus an unobserved shock.
    Neither the generator probability nor the shock is returned as a feature.
    Each milestone has exactly one snapshot, so no repeated entity crosses splits.
    """
    rng = np.random.default_rng(seed)
    as_of = pd.Timestamp("2024-01-01", tz="UTC") + pd.to_timedelta(rng.integers(0, 821, count), unit="D")
    days = rng.integers(1, 31, count)
    fails = rng.binomial(1, .18, count)
    info = (1 - fails) * rng.binomial(1, .25, count)
    unresolved = rng.binomial(1, .09, count)
    review = np.maximum(unresolved, rng.binomial(1, .28, count))
    not_started = rng.binomial(1, .30, count)
    at_risk = (1 - not_started) * rng.binomial(1, .22, count)
    open_fraction = rng.choice([0., .25, .5, .75, 1.], count)
    log_cost = rng.uniform(np.log1p(150000), np.log1p(5000000), count)
    log_odds = (-1.8 + 1.4 * fails + .65 * info + .7 * review + .8 * unresolved
                + .9 * not_started + .8 * at_risk + 1.5 * open_fraction - .045 * days
                + .12 * (log_cost - 13.7) + rng.normal(0, .55, count))
    late = rng.binomial(1, sigmoid(log_odds))
    delay = np.where(late == 1, rng.integers(1, 22, count), -rng.integers(0, 4, count))
    planned = as_of + pd.to_timedelta(days, unit="D")
    actual = planned + pd.to_timedelta(delay, unit="D")
    result = pd.DataFrame(dict(milestone_id=[f"SYN-M{i:05d}" for i in range(count)],
        prediction_as_of=as_of.strftime("%Y-%m-%dT%H:%M:%S+00:00"),
        planned_date=planned.strftime("%Y-%m-%dT%H:%M:%S+00:00"),
        actual_completion_date=actual.strftime("%Y-%m-%dT%H:%M:%S+00:00"),
        label_available_at=pd.DatetimeIndex(np.maximum(planned.asi8, actual.asi8), tz="UTC").strftime("%Y-%m-%dT%H:%M:%S+00:00"),
        days_to_planned_date=days, open_po_fraction=open_fraction, inspection_fail_flag=fails,
        inspection_info_flag=info, supplier_review_flag=review, unresolved_supplier_flag=unresolved,
        milestone_not_started_flag=not_started, milestone_at_risk_flag=at_risk, log_committed_cost=log_cost,
        late_flag=late, data_origin="SYNTHETIC_TRAINING_ONLY", generator_version="seer-history-v1", generator_seed=seed))
    feature_matrix(result)
    return result.sort_values(["prediction_as_of", "milestone_id"]).reset_index(drop=True)

history = synthetic_history(config["synthetic_seed"], config["synthetic_rows"])
if len(history) > config["max_driver_rows"]:
    raise ValueError("Training fixture exceeds the driver row limit")
display(history[["milestone_id", "prediction_as_of", "planned_date", "actual_completion_date", "late_flag", "data_origin"]].head())


## Task 3: Fix the data splits

Run the split cell and inspect its counts:

| Partition | Rule | Rows |
| --- | --- | --- |
| Training | Snapshot and outcome available before 2025-07-01 | 1,140 |
| Validation | Snapshot from 2025-07-01 to 2026-01-01; outcome available before 2026-01-01 | 391 |
| Test | Snapshot on or after 2026-01-01 | 183 |
| Excluded | Outcome not available by the relevant training/validation boundary | 86 |

Both rounds use these same partitions. Average precision summarizes precision-recall performance; higher is better. Brier score measures probability error; lower is better. Precision and recall are shown at the same fixed threshold for every candidate.

**Checkpoint:** with the supplied seed and 1,800-row configuration, the counts match this table. The 86 excluded rows are intentionally left out to prevent learning from outcomes that were unavailable at a split boundary. Do not change the partitions between training rounds.


In [ ]:
def chronological_split(history):
    as_of = pd.to_datetime(history.prediction_as_of, utc=True)
    available = pd.to_datetime(history.label_available_at, utc=True)
    validation_start = pd.Timestamp("2025-07-01", tz="UTC")
    test_start = pd.Timestamp("2026-01-01", tz="UTC")
    train = history.loc[(as_of < validation_start) & (available < validation_start)].copy()
    validation = history.loc[(as_of >= validation_start) & (as_of < test_start) & (available < test_start)].copy()
    test = history.loc[as_of >= test_start].copy()
    for name, frame in [("train", train), ("validation", validation), ("test", test)]:
        if len(frame) < 50 or frame.late_flag.nunique() != 2:
            raise ValueError(f"Insufficient rows or class coverage in {name}")
    return train, validation, test

train, validation, test = chronological_split(history)
X_train, y_train = train[FEATURES].astype(float), train.late_flag.astype(int)
X_validation, y_validation = validation[FEATURES].astype(float), validation.late_flag.astype(int)
# Reserve test rows; they are only used after choosing a winner.
X_test, y_test = test[FEATURES].astype(float), test.late_flag.astype(int)
DATASET_ID = hashlib.sha256(history.to_json(orient="records").encode()).hexdigest()
print("Rows:", {"train": len(train), "validation": len(validation), "test": len(test),
                "excluded_at_boundaries": len(history) - len(train) - len(validation) - len(test)})

def classification_metrics(labels, probabilities):
    predicted = probabilities >= threshold
    return {
        "average_precision": float(average_precision_score(labels, probabilities)),
        "roc_auc": float(roc_auc_score(labels, probabilities)),
        "precision": float(precision_score(labels, predicted, zero_division=0)),
        "recall": float(recall_score(labels, predicted, zero_division=0)),
        "f1": float(f1_score(labels, predicted, zero_division=0)),
        "brier_score": float(brier_score_loss(labels, probabilities)),
    }


## Task 4: Define run logging

Run the function-definition cell. It does not train a model yet.

Read `train_and_log`: it opens a run, logs the estimator parameters and data/split metadata, fits only training rows, evaluates validation rows, and logs a model artifact with its input signature and sample input.

The artifact's probability interface is used later for batch scoring. Model serialization/dependency files belong to MLflow-managed artifacts, not a learner `lib/` folder.

**Checkpoint:** the cell completes without displaying a trained model. This is expected: it defines `train_and_log` and `compare_runs`; Task 5 calls them. The `session_id` tag groups this walkthrough's runs, while `round` and `algorithm` distinguish the candidates.


In [ ]:
def train_and_log(run_name, algorithm, round_number, parameters):
    if algorithm == "Gradient Boosting":
        estimator = GradientBoostingClassifier(
            random_state=config["synthetic_seed"], **parameters)
    else:
        estimator = DecisionTreeClassifier(random_state=config["synthetic_seed"], **parameters)
    with mlflow.start_run(experiment_id=experiment.experiment_id, run_name=run_name) as run:
        mlflow.set_tags({"session_id": SESSION_ID, "round": str(round_number),
                         "algorithm": algorithm, "dataset_id": DATASET_ID,
                         "training_origin": "SYNTHETIC_TRAINING_ONLY"})
        mlflow.log_params({**estimator.get_params(), "train_end": "2025-07-01",
                           "validation_end": "2026-01-01", "threshold": threshold,
                           "training_rows": len(train), "validation_rows": len(validation)})
        estimator.fit(X_train, y_train)
        probabilities = estimator.predict_proba(X_validation)[:, 1]
        mlflow.log_metrics({"validation_" + key: value
                            for key, value in classification_metrics(y_validation, probabilities).items()})
        # Log probability output, not only the predicted class.
        model_info = mlflow.sklearn.log_model(
            sk_model=estimator, name="model",
            signature=infer_signature(X_train, estimator.predict_proba(X_train)),
            input_example=X_train.head(3), pyfunc_predict_fn="predict_proba",
            pip_requirements=[f"scikit-learn=={sklearn.__version__}",
                              f"pandas=={pd.__version__}", f"numpy=={np.__version__}"])
        mlflow.set_tag("model_uri", model_info.model_uri)
        print(run_name, "| run:", run.info.run_id,
              "| validation average precision:", round(classification_metrics(y_validation, probabilities)["average_precision"], 4))
    return run.info.run_id

def compare_runs(round_number=None):
    selected_filter = f"tags.session_id = '{SESSION_ID}'"
    if round_number is not None:
        selected_filter += f" AND tags.round = '{round_number}'"
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id],
                              filter_string=selected_filter, max_results=100)
    runs = runs.loc[runs.status == "FINISHED"].copy()
    return runs.sort_values(["metrics.validation_average_precision", "metrics.validation_brier_score", "run_id"],
                            ascending=[False, True, True]).reset_index(drop=True)

comparison_columns = ["run_id", "tags.mlflow.runName", "tags.round", "tags.algorithm",
                      "metrics.validation_average_precision", "metrics.validation_recall",
                      "metrics.validation_precision", "metrics.validation_f1", "metrics.validation_brier_score"]
# Friendly identifiers keep AIDP's table display from renaming dotted columns.
comparison_labels = {name: name.replace("tags.", "").replace("metrics.", "").replace("mlflow.", "") for name in comparison_columns}


## Task 5: Round one — Gradient Boosting

Run the three-candidate cell.

| Candidate | max_depth | learning_rate | n_estimators |
| --- | --- | --- | --- |
| gb_r1_1 | 2 | 0.05 | 60 |
| gb_r1_2 | 3 | 0.10 | 100 |
| gb_r1_3 | 4 | 0.05 | 120 |

Each candidate gets its own run in the same experiment. All are trained from scratch.

`max_depth` limits each tree's complexity; `learning_rate` controls each boosting step's contribution; `n_estimators` is the number of boosting trees. Gradient Boosting is provided by the existing scikit-learn runtime; it is not XGBoost and is not a pretrained model.

**Checkpoint:** three printed run names, three distinct run IDs and three validation average-precision values. Allow artifact logging to finish; a completed model fit alone does not mean its run has finished.

Run this training round once per session. If it partially fails or was repeated, see the rerun guidance at the start of this notebook before retrying.


In [ ]:
gb_candidates = [
    {"max_depth": 2, "learning_rate": 0.05, "n_estimators": 60},
    {"max_depth": 3, "learning_rate": 0.10, "n_estimators": 100},
    {"max_depth": 4, "learning_rate": 0.05, "n_estimators": 120},
]
round_one_ids = [train_and_log(f"gb_r1_{index}", "Gradient Boosting", 1, parameters)
                 for index, parameters in enumerate(gb_candidates, start=1)]


## Task 6: Compare round one

1. Run the notebook comparison cell. It queries the experiment service.
2. In AIDP, open **Experiments → seer_milestone_delay** and refresh.
3. Use **List** to inspect the three run names. On repeated exercises, use tags/run details to identify this session.
4. Open a run to inspect Parameters, Metrics and model artifacts.
5. Switch to **Compare** to inspect the available metric charts.
6. Use the notebook table for the exact session-filtered comparison if the UI does not expose a tag filter.
7. Discuss the tradeoff between recall and precision; do not inspect the test set to decide which candidate to prefer.

**Checkpoint:** the notebook comparison has exactly three rows for this session, ranked by validation average precision. If you have run the lab before, the UI may show additional historical runs with the same names; use run IDs and the notebook's session-filtered results to distinguish them. Do not delete older runs to make the table match.


In [ ]:
initial_comparison = compare_runs(round_number=1)
display(initial_comparison[comparison_columns].rename(columns=comparison_labels))


## Task 7: Round two — Decision Tree

Return to the notebook and run the three Decision Tree candidates.

| Candidate | max_depth | min_samples_leaf |
| --- | --- | --- |
| dt_r2_1 | 3 | 10 |
| dt_r2_2 | 5 | 20 |
| dt_r2_3 | 8 | 30 |

Keep the same experiment and data. Refresh the experiment page: both rounds remain available together.


The three `gb_r1_` runs and three `dt_r2_` runs belong to the same experiment. Test metrics appear only on the selected winner after Task 8.

`min_samples_leaf` is the minimum number of training examples in a leaf; increasing it discourages very small, highly specific branches. Unlike boosting, each Decision Tree candidate fits one tree.

**Checkpoint:** three new run IDs are printed. Keep both rounds: do not create a second experiment or rerun Task 1 between them.

Run this training round once per session. If it partially fails or was repeated, see the rerun guidance at the start of this notebook before retrying.


In [ ]:
tree_candidates = [
    {"max_depth": 3, "min_samples_leaf": 10},
    {"max_depth": 5, "min_samples_leaf": 20},
    {"max_depth": 8, "min_samples_leaf": 30},
]
round_two_ids = [train_and_log(f"dt_r2_{index}", "DecisionTree", 2, parameters)
                 for index, parameters in enumerate(tree_candidates, start=1)]


## Task 8: Compare all six and select the winner


1. Run the final comparison cell.
2. Confirm it contains exactly the six run IDs from this exercise.
3. The ranking uses descending validation average precision, ascending Brier score, then run ID for stable tie-breaking.
4. Note the winning run ID and algorithm.
5. The cell loads the winning artifact and evaluates it once on the reserved test partition.
6. Test metrics are added to the winning run. They were not used to select the winner.
7. The winner can change when library versions, parameters or data change. Do not hard-code a winning algorithm.

**Checkpoint:** six rows appear in the notebook comparison, followed by the selected run ID, algorithm and test metrics. Use `validation_average_precision` for selection—not the highest recall or a test metric. The first verified run selected `gb_r1_1`; that is an illustration, not a required outcome after changing inputs.


In [ ]:
final_comparison = compare_runs()
if set(final_comparison.run_id) != set(round_one_ids + round_two_ids):
    raise RuntimeError("All six candidates must finish before selection")
display(final_comparison[comparison_columns].rename(columns=comparison_labels))
best_run_id = final_comparison.iloc[0]["run_id"]
best_run = client.get_run(best_run_id)
best_model_uri = best_run.data.tags["model_uri"]
best_model = mlflow.sklearn.load_model(best_model_uri)
test_metrics = classification_metrics(y_test, best_model.predict_proba(X_test)[:, 1])
with mlflow.start_run(run_id=best_run_id):
    mlflow.log_metrics({"test_" + key: value for key, value in test_metrics.items()})
    mlflow.set_tag("selected_by", "validation_average_precision_then_brier")
print("Selected run:", best_run_id, "| algorithm:", best_run.data.tags["algorithm"])
print("Untouched test metrics:", json.dumps(test_metrics, indent=2))


## Task 9: Register in the Master Catalog

1. Run the registration cell once.
2. Record the printed model name, numeric version, source run and versioned URI.
3. Open **Master catalog → seer_livelabs_20260922 → seer_gold → Models**.
4. Open **milestone_delay_classifier** and inspect its versions.
5. Follow the originating run to check the training parameters and metrics.
6. The alternative UI route is **Experiments → selected run → Register**. Choose the logged model and this catalog/schema. Do not also use this route after notebook registration unless you intentionally want another version.

Registration stores a catalog model asset, not another Gold table. It is not production approval.

On the model page, choose **Versions** and use the **Run** link to inspect the source run. The fully qualified name includes catalog, schema and model; registering only an unqualified short name is not this exercise.

**Checkpoint:** the printed model name matches the table above and the Versions tab contains the returned version. Version 1 in the screenshot is from the first verified execution; an existing model can return version 2 or later. Running this cell again creates another version, so do not rerun it just to refresh the UI.



In [ ]:
registered_version = mlflow.register_model(model_uri=best_model_uri, name=MODEL_NAME)
MODEL_VERSION = str(registered_version.version)
REGISTERED_URI = f"models:/{MODEL_NAME}/{MODEL_VERSION}"
print("Registered model:", MODEL_NAME)
print("Version:", MODEL_VERSION, "| Source run:", best_run_id)
print("Versioned model URI:", REGISTERED_URI)


## Task 10: Load the registered version and score

1. Run the load-and-score cell.
2. It loads `models:/<catalog>.<schema>.<model>/<version>`, not an in-memory estimator or “latest”.
3. An MLflow Spark UDF uses the same version to score the saved Gold milestone features.
4. Inspect the three predictions and the `model_name`, `model_version`, `model_run_id` and `scoring_run_id` columns.
5. The original Gold `run_id` remains the source-data run ID.
6. A bounded read-back check compares saved probabilities with the loaded model's predictions.
7. Inspect `seer_gold.milestone_delay_predictions` in the catalog.

LOW/MEDIUM/HIGH bands are workshop conventions. An old source snapshot remains old regardless of when it was scored.

**Checkpoint:** three persisted rows are displayed, each with the same selected model name/version and training-run ID. No read-back exception means the notebook's saved-score comparison passed. The code writes predictions from the registered model, not from the estimator left in memory after training.

| Output column | Meaning |
| --- | --- |
| `delay_probability` | Estimated delayed-class probability, from 0 to 1 |
| `predicted_late` | True when probability is at least 0.5 |
| `risk_band` | LOW below 0.4; MEDIUM from 0.4 to below 0.7; HIGH from 0.7 |
| `model_name`, `model_version` | Catalog model and exact version used for inference |
| `model_run_id` | Training run that produced the selected artifact |
| `run_id` | Source Gold data's run identifier |
| `scoring_run_id` | Current notebook/workflow scoring run identifier |


In [ ]:
loaded_model = mlflow.sklearn.load_model(REGISTERED_URI)
print("Loaded registered version:", REGISTERED_URI)
feature_rows = spark.table(f"{GOLD}.milestone_features")
probability_udf = mlflow.pyfunc.spark_udf(
    spark, model_uri=REGISTERED_URI, result_type="array<double>", env_manager="local")
# The logged predict_proba returns [probability_on_time, probability_delayed].
probability = probability_udf(F.struct(*[F.col(name).cast("double").alias(name) for name in FEATURES]))[1]
scored = feature_rows.select(
    "project_id", "asset_id", "milestone_id", "project_name", "milestone_name",
    "planned_date", "prediction_as_of", "committed_cost_cents",
    probability.alias("delay_probability"), "run_id")
scored = (scored.withColumn("predicted_late", F.col("delay_probability") >= threshold)
    .withColumn("risk_band", F.when(F.col("delay_probability") < .4, "LOW")
        .when(F.col("delay_probability") < .7, "MEDIUM").otherwise("HIGH"))
    .withColumn("model_name", F.lit(MODEL_NAME))
    .withColumn("model_version", F.lit(MODEL_VERSION))
    .withColumn("model_run_id", F.lit(best_run_id))
    .withColumn("scoring_run_id", F.lit(RUN_ID))
    .withColumn("model_training_origin", F.lit("SYNTHETIC_TRAINING_ONLY"))
    .withColumn("prediction_scope", F.lit("HISTORICAL_FIXTURE_DEMONSTRATION_NOT_OPERATIONAL"))
    .withColumn("scored_at", F.lit(datetime.now(timezone.utc).isoformat())))
scored.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.milestone_delay_predictions")
prediction_rows = spark.table(f"{GOLD}.milestone_delay_predictions")
display(prediction_rows.orderBy(F.desc("delay_probability")))
# Bounded read-back check: registered-model scores must match saved Spark scores.
sample = feature_rows.limit(config["max_driver_rows"] + 1).toPandas()
if len(sample) > config["max_driver_rows"]:
    raise ValueError("Read-back check exceeds workshop driver row limit")
expected = pd.DataFrame({"milestone_id": sample.milestone_id,
                         "expected": loaded_model.predict_proba(sample[FEATURES].astype(float))[:, 1]})
actual = prediction_rows.select("milestone_id", "delay_probability").limit(config["max_driver_rows"] + 1).toPandas()
comparison = actual.merge(expected, on="milestone_id", validate="one_to_one")
if len(comparison) != len(expected) or not np.allclose(comparison.delay_probability, comparison.expected, rtol=1e-6, atol=1e-8):
    raise ValueError("Persisted predictions do not match the registered model")
context = spark.table(f"{GOLD}.project_context")


## Task 11: Stage the project PDFs

1. Run the PDF-staging cell after scoring.
2. The cell verifies the three source PDFs against the supplied manifest and copies missing files to the Austin folder. It stops for review if existing bytes differ.
3. Note the printed path: `/Volumes/seer_livelabs_20260922/seer_gold/workshop_outputs/austin-project`.
4. Open **Master catalog → seer_livelabs_20260922 → seer_gold → Volumes → workshop_outputs → austin-project** and confirm the three PDFs.

**Checkpoint:** the folder contains the supplier agreement, receiving inspection report and structural specification. Staging files does not create or ingest a knowledge base.


In [ ]:
source_root = Path(config["source_root"])
kb_source_folder = Path(config["output_root"]) / "austin-project"
kb_source_folder.mkdir(parents=True, exist_ok=True)
documents = json.loads((LAB_HOME / "reference/pdf-manifest.json").read_text())
expected_names = {item["document_name"] for item in documents}
assert len(expected_names) == 3
assert all(path.name in expected_names for path in kb_source_folder.iterdir()), "Unexpected file in KB folder"
for item in documents:
    relative = "documents/" + item["document_name"]
    source = source_root / (relative.replace("/", "_") if config["source_layout"] == "flat" else relative)
    target = kb_source_folder / item["document_name"]
    assert hashlib.sha256(source.read_bytes()).hexdigest() == item["sha256"], source
    if target.exists():
        assert hashlib.sha256(target.read_bytes()).hexdigest() == item["sha256"], "Review the existing PDF before replacing it"
    else:
        shutil.copyfile(source, target)
    assert hashlib.sha256(target.read_bytes()).hexdigest() == item["sha256"]
print("Knowledge-base source folder:", kb_source_folder)
